# ARC3 isolated training dependency probe

CPU only. Reuses the frozen offline runtime; adds pinned PEFT/Accelerate wheels without changing Torch/CUDA/Transformers. Meta-model construction does not establish training or model quality. No game access or weight loading. Qwen, Hugging Face and Pennyroyal attribution retained.

In [ ]:
PACKAGES={'peft': {'version': '0.21.2', 'filename': 'peft-0.21.2-py3-none-any.whl', 'sha256': '106ab6077ff72c54d21577f9af5970e34bac7582cb14209f7b2b511e322a4eae'}, 'accelerate': {'version': '1.15.0', 'filename': 'accelerate-1.15.0-py3-none-any.whl', 'sha256': '97eacca0b73e45cb867dbf8c5d5d4dc32219544300e0c8992c7334dc2ef33cec'}}
MODEL_CONFIG={'architectures': ['Qwen3_5ForConditionalGeneration'], 'image_token_id': 248056, 'language_model_only': False, 'model_type': 'qwen3_5', 'text_config': {'attention_bias': False, 'attention_dropout': 0.0, 'attn_output_gate': True, 'bos_token_id': 248044, 'dtype': 'bfloat16', 'eos_token_id': 248044, 'full_attention_interval': 4, 'head_dim': 256, 'hidden_act': 'silu', 'hidden_size': 5120, 'initializer_range': 0.02, 'intermediate_size': 17408, 'layer_types': ['linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention', 'linear_attention', 'linear_attention', 'linear_attention', 'full_attention'], 'linear_conv_kernel_dim': 4, 'linear_key_head_dim': 128, 'linear_num_key_heads': 16, 'linear_num_value_heads': 48, 'linear_value_head_dim': 128, 'mamba_ssm_dtype': 'float32', 'max_position_embeddings': 262144, 'model_type': 'qwen3_5_text', 'mtp_num_hidden_layers': 1, 'mtp_use_dedicated_embeddings': False, 'num_attention_heads': 24, 'num_hidden_layers': 64, 'num_key_value_heads': 4, 'output_gate_type': 'swish', 'pad_token_id': None, 'partial_rotary_factor': 0.25, 'rms_norm_eps': 1e-06, 'rope_parameters': {'mrope_interleaved': True, 'mrope_section': [11, 11, 10], 'partial_rotary_factor': 0.25, 'rope_theta': 10000000, 'rope_type': 'default'}, 'tie_word_embeddings': False, 'use_cache': True, 'vocab_size': 248320}, 'tie_word_embeddings': False, 'transformers_version': '5.8.0.dev0', 'video_token_id': 248057, 'vision_config': {'deepstack_visual_indexes': [], 'depth': 27, 'hidden_act': 'gelu_pytorch_tanh', 'hidden_size': 1152, 'in_channels': 3, 'initializer_range': 0.02, 'intermediate_size': 4304, 'model_type': 'qwen3_5', 'num_heads': 16, 'num_position_embeddings': 2304, 'out_hidden_size': 5120, 'patch_size': 16, 'spatial_merge_size': 2, 'temporal_patch_size': 2}, 'vision_end_token_id': 248054, 'vision_start_token_id': 248053}
BASE_LOCK_SHA256='4b0ee12848c1e7c3a51b0819de3a1ae63a967313fd1f62b9e2ca6b9c82b8da02'
from pathlib import Path
from hashlib import sha256
import json,os,subprocess,sys,zipfile,urllib.request
work=Path('/kaggle/working');bundle=work/'arc3-training-dependencies';bundle.mkdir(exist_ok=True)
roots=[Path('/kaggle/input/datasets/dfranzen/pennyroyal-v253'),Path('/kaggle/input/pennyroyal-v253')]
root=next(p for p in roots if p.is_dir());wheels=root/'wheels';lock=root/'requirements.lock'
assert sha256(lock.read_bytes()).hexdigest()==BASE_LOCK_SHA256,'Base lock changed'
environment=dict(os.environ,UV_OFFLINE='1',UV_PYTHON_DOWNLOADS='never',PYTHONNOUSERSITE='1',
    HF_HUB_OFFLINE='1',TRANSFORMERS_OFFLINE='1',USE_TF='0',USE_TORCHVISION='0')
environment.pop('PYTHONPATH',None)
venv=Path('/tmp/arc3-training-cpu-env');uv=Path('/tmp/arc3-training-uv');python=venv/'bin/python'
uv_wheels=list(wheels.glob('uv-*.whl'));assert len(uv_wheels)==1
with zipfile.ZipFile(uv_wheels[0]) as archive:
    names=[n for n in archive.namelist() if Path(n).name=='uv' and not n.endswith('/')];assert len(names)==1
    uv.write_bytes(archive.read(names[0]));uv.chmod(0o755)
def run(arguments):
    result=subprocess.run(list(map(str,arguments)),env=environment,text=True,capture_output=True)
    print(result.stdout[-3500:],flush=True)
    if result.returncode:print(result.stderr[-7000:],flush=True);raise RuntimeError('Isolated CPU command failed')
run([uv,'venv','--python',sys.executable,venv])
run([uv,'pip','install','--python',python,'--no-index','--find-links',wheels,'-r',lock])
# Public, pinned pure Python wheels only. Existing Torch/CUDA/Transformers are retained.
for name,package in PACKAGES.items():
    with urllib.request.urlopen('https://pypi.org/pypi/'+name+'/'+package['version']+'/json',timeout=30) as stream:
        current=json.load(stream)
    wheel=next(x for x in current['urls'] if x['filename']==package['filename'])
    with urllib.request.urlopen(wheel['url'],timeout=60) as stream:content=stream.read()
    assert sha256(content).hexdigest()==package['sha256'],'Pinned wheel mismatch'
    (bundle/package['filename']).write_bytes(content)
run([uv,'pip','install','--python',python,'--no-index','--no-deps',*sorted(bundle.glob('*.whl'))])
run([uv,'pip','check','--python',python])
config_path=work/'qwen27-bf16-config.json';config_path.write_text(json.dumps(MODEL_CONFIG))
probe=work/'training_meta_probe.py'
probe.write_text(r"""import json,torch,transformers,peft,accelerate,inspect
from pathlib import Path
from transformers import AutoConfig,AutoModelForImageTextToText
from accelerate import init_empty_weights
raw=json.loads(Path('/kaggle/working/qwen27-bf16-config.json').read_text())
kind=raw.pop('model_type');config=AutoConfig.for_model(kind,**raw)
with init_empty_weights():
    model=AutoModelForImageTextToText.from_config(config,torch_dtype=torch.bfloat16)
modules=[]
for name,module in model.named_modules():
    if isinstance(module,torch.nn.Linear) and 'language_model.layers.' in name:
        modules.append(dict(name=name,in_features=module.in_features,out_features=module.out_features))
assert modules,'No language-model linear modules found'
assert all(p.device.type=='meta' for p in model.parameters()),'Unexpected real parameter allocation'
report=dict(passed=True,torch=torch.__version__,transformers=transformers.__version__,peft=peft.__version__,
    accelerate=accelerate.__version__,cuda_available=torch.cuda.is_available(),
    model_class=type(model).__name__,supports_gradient_checkpointing=model.supports_gradient_checkpointing,
    linear_modules=modules,parameters=sum(p.numel() for p in model.parameters()),
    boundary='CPU imports and meta-device model construction only. No loaded checkpoint, multimodal tensor validation, actual gradients, CUDA training, saved-adapter reload or serving compatibility established.')
Path('/kaggle/working/training-environment-probe.json').write_text(json.dumps(report,indent=2)+'\n')
print(json.dumps({k:v for k,v in report.items() if k!='linear_modules'},indent=2))
""")
run([python,probe])
manifest=dict(packages=PACKAGES,base_runtime='dfranzen/pennyroyal-v253',base_lock_sha256=BASE_LOCK_SHA256,
    files={p.name:sha256(p.read_bytes()).hexdigest() for p in sorted(bundle.glob('*.whl'))})
(bundle/'manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
print('TRAINING_DEPENDENCY_CPU_CHECK_COMPLETE',flush=True)
